In [1]:
import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
import psutil
import time
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
import torch
import torch.nn as nn

class DLRM(nn.Module):
    def __init__(self, num_dense_features, vocab_sizes, embed_dim, bottom_mlp_dims, top_mlp_dims):
        super(DLRM, self).__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(vocab, embed_dim) for vocab in vocab_sizes])
        
        # Bottom MLP xử lý Dense features
        bottom_layers = []
        in_dim = num_dense_features
        for dim in bottom_mlp_dims:
            bottom_layers.append(nn.Linear(in_dim, dim))
            bottom_layers.append(nn.ReLU())
            in_dim = dim
        self.bottom_mlp = nn.Sequential(*bottom_layers)
        
        assert bottom_mlp_dims[-1] == embed_dim, "Output của Bottom MLP phải bằng với embed_dim"
        
        # input Top MLP
        num_sparse = len(vocab_sizes)
        #  output dense + các cặp tương tác (dot products)
        interaction_dim = embed_dim + ((num_sparse + 1) * num_sparse) // 2
        
        # Top MLP
        top_layers = []
        in_dim = interaction_dim
        for dim in top_mlp_dims:
            top_layers.append(nn.Linear(in_dim, dim))
            top_layers.append(nn.ReLU())
            in_dim = dim
        top_layers.append(nn.Linear(in_dim, 1))
        self.top_mlp = nn.Sequential(*top_layers)

    def forward(self, dense_x, sparse_x):
        dense_out = self.bottom_mlp(dense_x) # (batch, embed_dim)
        
        emb_x = [emb(sparse_x[:, i]) for i, emb in enumerate(self.embeddings)]
        
        emb_x.append(dense_out) 
        stacked_emb = torch.stack(emb_x, dim=1) # (batch, num_sparse + 1, embed_dim)
        
        interaction_matrix = torch.bmm(stacked_emb, stacked_emb.transpose(1, 2))
        
        num_features = stacked_emb.size(1)
        row_idx, col_idx = torch.triu_indices(num_features, num_features, offset=1)
        interactions = interaction_matrix[:, row_idx, col_idx]
        
        # original dense_out concat vector interactions
        concat_x = torch.cat([dense_out, interactions], dim=1)
        
        # 5. Đi qua Top MLP
        out = self.top_mlp(concat_x)
        return  out.squeeze(1)

In [ ]:
import math
from torch.utils.data import Dataset
import torch
from tqdm.auto import tqdm

class StandardCriteoDataset(Dataset):
    def __init__(self, hf_dataset, dense_cols, cat_cols, cat_mappers=None, block_vocab_sizes=None):
        self.data = hf_dataset
        self.dense_cols = dense_cols
        self.cat_cols = cat_cols
        
        if cat_mappers is None or block_vocab_sizes is None:
            print("Đang đếm Vocabulary Size cho tập TRAIN...")
            self.block_vocab_sizes = []
            self.cat_mappers = []
            self._build_vocab()
        else:
            print("Sử dụng Vocabulary đã học từ tập TRAIN...")
            self.cat_mappers = cat_mappers
            self.block_vocab_sizes = block_vocab_sizes
            
    def _build_vocab(self):
        df = self.data.to_pandas()
        for col in tqdm(self.cat_cols, desc="Building Vocab"):
            unique_vals = df[col].dropna().unique()
            mapper = {val: idx + 1 for idx, val in enumerate(unique_vals)}
            self.cat_mappers.append(mapper)
            self.block_vocab_sizes.append(len(unique_vals) + 1)
            
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        
        dense_x = []
        for c in self.dense_cols:
            val = row[c]
            if val is None or val < 0:
                dense_x.append(0.0)
            else:
                dense_x.append(math.log1p(float(val))) 
        
        # Categorical
        cat_ids = []
        for col_idx, col_name in enumerate(self.cat_cols):
            val = row[col_name]
            mapper = self.cat_mappers[col_idx]
            cat_ids.append(mapper.get(val, 0)) 
            
        label = float(row['label'])

        return (
            torch.tensor(dense_x, dtype=torch.float32),
            torch.tensor(cat_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32)
        )

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
import os

dense_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

train_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/train.parquet"
val_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/validation.parquet"

train_hf = load_dataset("parquet", data_files=train_file, split="train")
val_hf = load_dataset("parquet", data_files=val_file, split="train")

print(f"Số mẫu -> Train: {len(train_hf)} | Val: {len(val_hf)}")

print("Đang khởi tạo Dataset (Xây dựng Vocab trên Train)...")
train_dataset = StandardCriteoDataset(train_hf, dense_cols, cat_cols)

train_mappers = train_dataset.cat_mappers
train_vocab_sizes = train_dataset.block_vocab_sizes

print("Đang khởi tạo Val Dataset...")
val_dataset = StandardCriteoDataset(val_hf, dense_cols, cat_cols, cat_mappers=train_mappers, block_vocab_sizes=train_vocab_sizes)

print("Đang tạo DataLoader...")
train_loader = DataLoader(train_dataset, batch_size=8192, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_loader = DataLoader(val_dataset, batch_size=8192, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)

print("Hoàn tất tạo DataLoader!")

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/32 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Số mẫu -> Train: 36665662 | Val: 4583562
Đang khởi tạo Dataset (Xây dựng Vocab trên Train)...
Đang đếm Vocabulary Size cho tập TRAIN...


Building Vocab:   0%|          | 0/26 [00:00<?, ?it/s]

Đang khởi tạo Val Dataset...
Sử dụng Vocabulary đã học từ tập TRAIN...
Đang tạo DataLoader...
Hoàn tất tạo DataLoader!


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

model = DLRM(
    num_dense_features=len(dense_cols),
    vocab_sizes=train_vocab_sizes,
    embed_dim=64,
    bottom_mlp_dims=[256, 128, 64],
    top_mlp_dims=[256, 128, 64]
)

if num_gpus > 1:
    print(f"Dùng DataParallel trên {num_gpus} GPU")
    model = nn.DataParallel(model)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

Số GPU khả dụng: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
print("Bắt đầu huấn luyện SE-DLRM...")
EPOCHS = 5
best_val_auc = 0.0
save_path = "best_se_dcn_model.pth"

torch.cuda.reset_peak_memory_stats()

for epoch in range(EPOCHS):
    # train
    model.train()
    train_loss = 0.0
    total_ram_gb = 0.0
    total_vram_gb = 0.0
    batch_count = 0

    train_bar = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{EPOCHS}")

    for dense_x, cat_x, labels in train_bar:
        dense_x = dense_x.to(device, non_blocking=True)
        cat_x   = cat_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(dense_x, cat_x)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        
        # Đo lường tài nguyên
        current_ram = psutil.virtual_memory().used / (1024**3)
        current_vram = torch.cuda.memory_allocated() / (1024**3)
        total_ram_gb += current_ram
        total_vram_gb += current_vram
        batch_count += 1

        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)

    # val
    model.eval()
    val_loss = 0.0
    val_targets, val_preds = [], []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}/{EPOCHS}", leave=False)
        for dense_x, cat_x, labels in val_bar:
            dense_x = dense_x.to(device, non_blocking=True)
            cat_x   = cat_x.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)

            logits = model(dense_x, cat_x)
            loss   = criterion(logits, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            
            val_targets.append(labels.cpu().numpy())
            val_preds.append(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    
    val_targets_arr = np.concatenate(val_targets)
    val_preds_arr = np.concatenate(val_preds)
    val_auc = roc_auc_score(val_targets_arr, val_preds_arr)

    # Thống kê
    avg_ram = total_ram_gb / batch_count
    avg_vram = total_vram_gb / batch_count
    max_vram = torch.cuda.max_memory_allocated() / (1024**3)

    print(f"Tài nguyên Epoch {epoch+1}:")
    print(f"- RAM trung bình:  {avg_ram:.2f} GB")
    print(f"- VRAM trung bình: {avg_vram:.2f} GB | Đỉnh điểm: {max_vram:.2f} GB")

    print(
        f"Epoch {epoch+1}: \n"
        f"Train Loss: {avg_train_loss:.4f} || "
        f"Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}\n"
    )

    if val_auc > best_val_auc:
        print(f"Best Val AUC: {val_auc:.4f}. Đang lưu model...")
        best_val_auc = val_auc
        torch.save(model.state_dict(), save_path)
    else:
        print("Val AUC không cải thiện.")
    
    del val_targets, val_preds, val_targets_arr, val_preds_arr
    gc.collect()

Bắt đầu huấn luyện SE-DLRM...


[TRAIN] Epoch 1/5: 100%|██████████| 4476/4476 [10:07<00:00,  7.36it/s, Loss=0.4502]


Tài nguyên Epoch 1:
- RAM trung bình:  22.17 GB
- VRAM trung bình: 26.95 GB | Đỉnh điểm: 33.68 GB
Epoch 1: 
Train Loss: 0.4536 || Val Loss: 0.4460 | Val AUC: 0.8058

Best Val AUC: 0.8058. Đang lưu model...


[TRAIN] Epoch 2/5: 100%|██████████| 4476/4476 [10:04<00:00,  7.40it/s, Loss=0.3870]


Tài nguyên Epoch 2:
- RAM trung bình:  28.56 GB
- VRAM trung bình: 26.95 GB | Đỉnh điểm: 33.68 GB
Epoch 2: 
Train Loss: 0.3794 || Val Loss: 0.4980 | Val AUC: 0.7742

Val AUC không cải thiện.


...

Tài nguyên Epoch 3:
- RAM trung bình:  28.55 GB
- VRAM trung bình: 26.95 GB | Đỉnh điểm: 33.68 GB
Epoch 3: 
Train Loss: 0.3203 || Val Loss: 0.5647 | Val AUC: 0.7490

Val AUC không cải thiện.


[TRAIN] Epoch 4/5: 100%|██████████| 4476/4476 [10:04<00:00,  7.41it/s, Loss=0.2912]


Tài nguyên Epoch 4:
- RAM trung bình:  28.55 GB
- VRAM trung bình: 26.95 GB | Đỉnh điểm: 33.68 GB
Epoch 4: 
Train Loss: 0.2885 || Val Loss: 0.6029 | Val AUC: 0.7415

Val AUC không cải thiện.


[TRAIN] Epoch 5/5: 100%|██████████| 4476/4476 [10:04<00:00,  7.40it/s, Loss=0.2886]


Tài nguyên Epoch 5:
- RAM trung bình:  28.53 GB
- VRAM trung bình: 26.95 GB | Đỉnh điểm: 33.68 GB
Epoch 5: 
Train Loss: 0.2740 || Val Loss: 0.6389 | Val AUC: 0.7363

Val AUC không cải thiện.


In [ ]:
print("Bắt đầu đánh giá trên tập TEST và đo Latency...")

test_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/test.parquet"
test_hf = load_dataset("parquet", data_files=test_file, split="train")
test_dataset = StandardCriteoDataset(test_hf, dense_cols, cat_cols, cat_mappers=train_mappers, block_vocab_sizes=train_vocab_sizes)
test_loader = DataLoader(
    test_dataset, 
    batch_size=8192, 
    shuffle=False, 
    num_workers=8, 
    pin_memory=True, 
    persistent_workers=True, 
    prefetch_factor=2
)

model.load_state_dict(torch.load("best_se_dcn_model.pth", map_location=device))
model.eval()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

total_latency_ms = 0.0
total_samples = 0
latency_batch_count = 0
WARMUP_BATCHES = 5 

test_loss = 0.0
test_targets, test_preds = [], []

with torch.no_grad():
    test_bar = tqdm(test_loader, desc="[TEST] Evaluating")
    for i, (dense_x, cat_x, labels) in enumerate(test_bar):
        dense_x = dense_x.to(device, non_blocking=True)
        cat_x   = cat_x.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True)

        batch_size = dense_x.size(0)

        # forward pass
        start_event.record()
        logits = model(dense_x, cat_x)
        end_event.record()
        
        loss = criterion(logits, labels)
        test_loss += loss.item()

        probs = torch.sigmoid(logits)
        test_targets.append(labels.cpu().numpy())
        test_preds.append(probs.cpu().numpy())
        
        # Đồng bộ và tính toán thời gian
        torch.cuda.synchronize()
        if i >= WARMUP_BATCHES:
            batch_latency = start_event.elapsed_time(end_event)
            total_latency_ms += batch_latency
            total_samples += batch_size
            latency_batch_count += 1

avg_test_loss = test_loss / len(test_loader)
test_targets_arr = np.concatenate(test_targets)
test_preds_arr = np.concatenate(test_preds)
test_auc = roc_auc_score(test_targets_arr, test_preds_arr)

if latency_batch_count > 0:
    avg_batch_latency = total_latency_ms / latency_batch_count
    avg_inference_per_sample = total_latency_ms / total_samples
else:
    avg_batch_latency = 0; avg_inference_per_sample = 0

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test AUC:  {test_auc:.4f}")

print(f"Latency 1 Batch (Size {test_loader.batch_size}): {avg_batch_latency:.2f} ms")
print(f"Latency 1 Dòng (Per Sample):      {avg_inference_per_sample:.4f} ms")

Bắt đầu đánh giá trên tập TEST và đo Latency...
Sử dụng Vocabulary đã học từ tập TRAIN...


[TEST] Evaluating: 100%|██████████| 561/561 [01:10<00:00,  7.95it/s]


Test Loss: 0.4456
Test AUC:  0.8058
Latency 1 Batch (Size 8192): 0.78 ms
Latency 1 Dòng (Per Sample):      0.0001 ms
